In [1]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv("socal2(in).csv")
print(df.head())

   image_id                 street             citi  n_citi  bed  bath  sqft  \
0         0  1317 Van Buren Avenue  Salton City, CA     317    3   2.0  1560   
1         1         124 C Street W      Brawley, CA      48    3   2.0   713   
2         2        2304 Clark Road     Imperial, CA     152    3   1.0   800   
3         3     755 Brawley Avenue      Brawley, CA      48    3   1.0  1082   
4         4  2207 R Carrillo Court     Calexico, CA      55    4   3.0  2547   

    price  
0  201900  
1  228500  
2  273950  
3  350000  
4  385100  


In [3]:
# remove image_id
df.drop(columns=["image_id"], inplace=True)
print(df.head())

                  street             citi  n_citi  bed  bath  sqft   price
0  1317 Van Buren Avenue  Salton City, CA     317    3   2.0  1560  201900
1         124 C Street W      Brawley, CA      48    3   2.0   713  228500
2        2304 Clark Road     Imperial, CA     152    3   1.0   800  273950
3     755 Brawley Avenue      Brawley, CA      48    3   1.0  1082  350000
4  2207 R Carrillo Court     Calexico, CA      55    4   3.0  2547  385100


In [4]:
# seperate street number and street name form street
df[['street_number', 'street_name']] = df['street'].str.extract(r'(\d+)\s(.+)')
df.drop(columns=['street'], inplace=True)



# Group rare street names
top_streets = df["street_name"].value_counts().index[:10]  # Keep only top 10 most common streets
df["street_name"] = df["street_name"].apply(lambda x: x if x in top_streets else "Other")

print(df.head())
print(df.shape)  # (num_rows, num_columns)


              citi  n_citi  bed  bath  sqft   price street_number street_name
0  Salton City, CA     317    3   2.0  1560  201900          1317       Other
1      Brawley, CA      48    3   2.0   713  228500           124       Other
2     Imperial, CA     152    3   1.0   800  273950          2304       Other
3      Brawley, CA      48    3   1.0  1082  350000           755       Other
4     Calexico, CA      55    4   3.0  2547  385100          2207       Other
(15474, 8)


In [5]:
# prepare data
df["price"] = np.log1p(df["price"])
# **REMOVE OUTLIERS HERE (AFTER CLEANING, BEFORE TRAINING)**
df = df[(df["price"] > df["price"].quantile(0.005)) & (df["price"] < df["price"].quantile(0.995))]

# Feature Engineering: **Add Interaction Features**
# Better interaction features
""""
df["bed_ratio"] = df["bed"] / df["sqft"]  # Bedrooms per sqft
df["bath_ratio"] = df["bath"] / df["sqft"]  # Bathrooms per sqft
df["bed_bath_ratio"] = df["bed"] / df["bath"]  # Bed-to-bath balance
"""
print(df.head())
X = df.drop(columns=["price"])  # Features
y = df["price"]  # Target variable



#print("Columns in X_train:", X_train.columns)
#print("Data Types in X_train:", X_train.dtypes)



              citi  n_citi  bed  bath  sqft      price street_number  \
0  Salton City, CA     317    3   2.0  1560  12.215533          1317   
1      Brawley, CA      48    3   2.0   713  12.339296           124   
2     Imperial, CA     152    3   1.0   800  12.520705          2304   
3      Brawley, CA      48    3   1.0  1082  12.765691           755   
4     Calexico, CA      55    4   3.0  2547  12.861261          2207   

  street_name  
0       Other  
1       Other  
2       Other  
3       Other  
4       Other  


In [8]:
from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import RobustScaler

# convert categorical columns into numerical values because XGBoost only works with numeric data
categorical_features = ["citi", "street_name"]
# categorical_features = ["location"]
noncategorical_features = ["n_citi", "bed", "bath", "sqft"]

# target encoding
target_enc = TargetEncoder()
df[categorical_features] = target_enc.fit_transform(df[categorical_features], df["price"])

print(df.head())
X = df[noncategorical_features + categorical_features]  # Features (numeric + encoded categorical)
y = df["price"]  # Target variable

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Cap ratios between -3 and 3 standard deviations
"""
X_train["bed_ratio"] = np.clip(X_train["bed_ratio"], -3, 3)
X_train["bath_ratio"] = np.clip(X_train["bath_ratio"], -3, 3)
X_train["bed_bath_ratio"] = np.clip(X_train["bed_bath_ratio"], -3, 3)
# 2. Scale ONLY the training data's ratios
scaler = RobustScaler()
X_train[["bed_ratio", "bath_ratio", "bed_bath_ratio"]] = scaler.fit_transform(X_train[["bed_ratio", "bath_ratio", "bed_bath_ratio"]])
X_test[["bed_ratio", "bath_ratio", "bed_bath_ratio"]] = scaler.transform(X_test[["bed_ratio", "bath_ratio", "bed_bath_ratio"]])
# one-hot encoding
"""
"""
encoder = OneHotEncoder(sparse_output=False, drop="first")  # drop="first" to prevent multicollinearity
# Transform categorical features
encoded_features = encoder.fit_transform(X[categorical_features])
# Get column names after encoding
encoded_feature_names = encoder.get_feature_names_out(categorical_features)
# Convert to DataFrame
df_encoded = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=X.index)
# Combine with numerical features
X_final = pd.concat([df_encoded, X[noncategorical_features]], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)

print(X_train.shape)  # Number of features after encoding """



        citi  n_citi  bed  bath  sqft      price street_number  street_name
0  13.261637     317    3   2.0  1560  12.215533          1317    13.336441
1  13.246256      48    3   2.0   713  12.339296           124    13.336441
2  13.227302     152    3   1.0   800  12.520705          2304    13.336441
3  13.246256      48    3   1.0  1082  12.765691           755    13.336441
4  13.271376      55    4   3.0  2547  12.861261          2207    13.336441


'\nencoder = OneHotEncoder(sparse_output=False, drop="first")  # drop="first" to prevent multicollinearity\n# Transform categorical features\nencoded_features = encoder.fit_transform(X[categorical_features])\n# Get column names after encoding\nencoded_feature_names = encoder.get_feature_names_out(categorical_features)\n# Convert to DataFrame\ndf_encoded = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=X.index)\n# Combine with numerical features\nX_final = pd.concat([df_encoded, X[noncategorical_features]], axis=1)\n\nX_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)\n\nprint(X_train.shape)  # Number of features after encoding '

In [ ]:
#for col in ["citi", "street_number", "street_name"]:
    #X_train[col] = X_train[col].astype("category")
    #X_test[col] = X_test[col].astype("category")

# XGBoost regressor
"""xgb_model = xgb.XGBRegressor(
    n_estimators=500,  # Number of trees (higher = better but slower)
    learning_rate=0.05,  # Step size (lower = more accurate but slower)
    max_depth=7,  # Depth of each tree (higher = complex model)
    objective="reg:squarederror",  # For regression tasks
    random_state=42,
    #enable_categorical=True  # New feature in XGBoost
)"""

xgb_model = xgb.XGBRegressor(
    n_estimators=700,       # More trees for better learning
    learning_rate=0.05,     # Lower learning rate for stable training
    max_depth=7,            # Prevents excessive depth (overfitting)
    subsample=0.8,          # Uses 80% of data per tree (reduces variance)
    colsample_bytree=0.8,   # Uses 80% of features per tree (improves generalization)
    reg_lambda=1,           # L2 regularization (reduces overfitting)
    reg_alpha=0.1,          # L1 regularization (adds sparsity)
    gamma=0.1,              # Minimum loss reduction required for a split
    random_state=42
)

# Step 2: Check for overfitting via Cross-Validation (NEW)
from sklearn.model_selection import cross_val_score
scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
print("Cross-Validation MAE:", -scores.mean())

# train model
xgb_model.fit(X_train, y_train)



In [ ]:
# Make Predictions and Evaluate the Model

# Predict on test set
y_pred = xgb_model.predict(X_test)

# Convert back to actual price
# y_pred = np.expm1(y_pred_log)  # Reverse the log transformation
# y_test_actual = np.expm1(y_test)  # Convert test labels back

# Evaluate performance
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.2f}")  # Measures how well model explains variance (closer to 1 is better)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Predict using XGBoost
y_pred_xgb = xgb_model.predict(X_test)

# Scatter plot
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred_xgb, alpha=0.5, color="blue", label="XGBoost")

# Add a diagonal line (perfect prediction reference)
x = np.linspace(min(y_test), max(y_test), 100)
plt.plot(x, x, color="red", linestyle="dashed", label="Perfect Predictions")

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("XGBoost Regression: Actual vs Predicted Prices")
plt.legend()
plt.show()


In [ ]:
# Understand Which Factors Affect Prices Most

import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we use the processed feature names from X_train
importance_df = pd.DataFrame({
    'Feature': X_train.columns,  # Use X_train.columns instead of X.columns
    'Importance': xgb_model.feature_importances_
})

# Sort by importance score
importance_df = importance_df.sort_values(by="Importance", ascending=False)

# Plot feature importance
plt.figure(figsize=(10,5))
sns.barplot(x=importance_df["Importance"], y=importance_df["Feature"])
plt.title("Feature Importance in House Price Prediction")
plt.show()



In [ ]:


# Get feature importance scores
feature_importance = xgb_model.feature_importances_

# Create DataFrame for visualization
importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by="Importance", ascending=False)

# Plot feature importance
plt.figure(figsize=(10,5))
sns.barplot(x=importance_df["Importance"], y=importance_df["Feature"])
plt.title("Feature Importance in House Price Prediction")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()


In [1]:
from graphviz import Digraph

# Sequence Diagram for VM-1 Scenario: Cappuccino Disposal

def create_vm1_sequence_diagram():
    dot = Digraph("VM1_Sequence")
    dot.attr(rankdir='LR')

    dot.node("user", "User", shape="actor")
    dot.node("vm1", "VM1")
    dot.node("mda", "MDA_EFSM")
    dot.node("op", "OpVM1")

    dot.edge("user", "vm1", "create(2.5)")
    dot.edge("vm1", "mda", "create()")
    dot.edge("mda", "op", "StorePrice()")

    dot.edge("user", "vm1", "insert_cups(20)")
    dot.edge("vm1", "mda", "insert_cups(20)")
    dot.edge("mda", "op", "ZeroCF()")

    dot.edge("user", "vm1", "card(7.2)")
    dot.edge("vm1", "mda", "card()")
    dot.edge("mda", "op", "ZeroCF()")

    dot.edge("user", "vm1", "sugar()")
    dot.edge("vm1", "mda", "additive(1)")
    dot.edge("mda", "op", "DisposeAdditive(A[])")

    dot.edge("user", "vm1", "cappuccino()")
    dot.edge("vm1", "mda", "dispose_drink(1)")
    dot.edge("mda", "op", "DisposeDrink(1)")
    dot.edge("mda", "op", "ZeroCF()")

    return dot

# Sequence Diagram for VM-2 Scenario: Coffee Disposal

def create_vm2_sequence_diagram():
    dot = Digraph("VM2_Sequence")
    dot.attr(rankdir='LR')

    dot.node("user", "User", shape="actor")
    dot.node("vm2", "VM2")
    dot.node("mda", "MDA_EFSM")
    dot.node("op", "OpVM2")

    dot.edge("user", "vm2", "CREATE(2)")
    dot.edge("vm2", "mda", "create()")
    dot.edge("mda", "op", "StorePrice()")

    dot.edge("user", "vm2", "InsertCups(1)")
    dot.edge("vm2", "mda", "insert_cups(1)")
    dot.edge("mda", "op", "ZeroCF()")

    dot.edge("user", "vm2", "COIN(1)")
    dot.edge("vm2", "mda", "coin(0)")
    dot.edge("mda", "op", "IncreaseCF()")

    dot.edge("user", "vm2", "COIN(1)")
    dot.edge("vm2", "mda", "coin(1)")
    dot.edge("mda", "op", "IncreaseCF()")

    dot.edge("user", "vm2", "CREAM()")
    dot.edge("vm2", "mda", "additive(1)")
    dot.edge("mda", "op", "DisposeAdditive(A[])")

    dot.edge("user", "vm2", "COFFEE()")
    dot.edge("vm2", "mda", "dispose_drink(1)")
    dot.edge("mda", "op", "DisposeDrink(1)")
    dot.edge("mda", "op", "ZeroCF()")

    return dot

# Render both diagrams
diagram_vm1 = create_vm1_sequence_diagram()
diagram_vm1.render("vm1_scenario", format="png", cleanup=True)

diagram_vm2 = create_vm2_sequence_diagram()
diagram_vm2.render("vm2_scenario", format="png", cleanup=True)


'vm2_scenario.png'